# KD Exploration

Trimmed exploratory notebook. All training/distillation logic now lives in `src/kd_pipeline` and `scripts/`, run via `dvc repro`. This notebook only loads the resulting artifacts and pokes at them interactively — it does not duplicate any pipeline logic.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kd_pipeline.config import load_params
from kd_pipeline.data import make_dataloaders
from kd_pipeline.model_io import load_model
from kd_pipeline.plotting import save_prediction_grid

params = load_params(Path.cwd().parent / "params.yaml")
params["distill"]

In [ ]:
# Requires the pipeline to have been run at least once (`dvc repro` from the repo root).
distilled_student = load_model(
    params["distill"]["student_arch"],
    Path.cwd().parent / params["distill"]["checkpoint"],
    device="cpu",
)
distilled_student

In [ ]:
data_cfg = params["data"]
_, _, test_loader, test_dataset = make_dataloaders(
    root=str(Path.cwd().parent / data_cfg["raw_dir"]),
    batch_size=data_cfg["batch_size"],
    num_workers=0,
    val_fraction=data_cfg["val_fraction"],
    seed=params["seed"],
    mean=data_cfg["normalize_mean"],
    std=data_cfg["normalize_std"],
    download=True,
)

save_prediction_grid(
    distilled_student,
    test_dataset,
    out_path="exploration_prediction_grid.png",
    num_samples=6,
    seed=123,
)

from IPython.display import Image
Image(filename="exploration_prediction_grid.png")